In [5]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_stx"


In [6]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 120 firm-variable mappings across 95 firms.


firm_id,variable,notes,final_choice,num_candidates
8TRA.ST,XSGA_COMPONENTS,"Candidate list appears to miss the important SG&A component 'Distribution expenses', which is present in the income statement and is a standard selling/distribution overhead line. No explicit single SG&A subtotal exists, so selected key overhead components. Excluded impairment, FX, legal/arbitration, restructuring-specific, and total/subtotal rows. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Distribution expenses",Income Statement::Distribution expenses; Income Statement::Administrative Expenses; Income Statement::Other operating expenses,8
AAK.ST,COGS,"Candidate list misses a key COGS component: 'Change in inventories of finished goods and work in progress', which is part of cost of sales/manufacturing expense. Added it from the full Income Statement preview, so manual review is required. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Change in inventories of finished goods and work in progress",Income Statement::Raw materials and consumables; Income Statement::Change in inventories of finished goods and work in progress; Income Statement::Goods for resale,4
ACELP.ST,REVT,Candidate list clearly misses the correct top-line label. Chose 'Net sales' from the Income Statement preview as the revenue row. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Net sales,Income Statement::Net sales,5
ACELP.ST,XSGA_COMPONENTS,"Used candidate rows where appropriate, but added 'Commercial preparation costs' and 'Other operating costs' from the full preview because they are relevant overhead components and the candidate list was incomplete. 'Commercial preparation costs' appears twice in the preview with the same label; included the label only once per deduplication rule. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Commercial preparation costs, [Income Statement] Other operating costs",Income Statement::Administrative costs; Income Statement::Commercial preparation costs; Income Statement::Other operating costs; Income Statement::Research and development costs,3
ADDTb.ST,REVT,"Selected candidate 'Total Revenue' as primary revenue line. Added 'Net sales' from the preview as fallback/priority alternative because it appears to be the same top-line concept and may be the label used in some periods. This second label is outside the candidate list, so manual review is flagged per instructions. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Net sales",Income Statement::Total Revenue; Income Statement::Net sales,15
AFRY.ST,XSGA_COMPONENTS,"Candidate list appears to miss the main populated overhead row 'Other costs', which is a better SG&A component than the mostly empty/tiny 'Other operating expenses'. Chose 'Personnel costs' from candidates plus out-of-list 'Other costs' from the preview; manual review needed to confirm 'Other costs' is recurring overhead and not mixed with excluded items. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Other costs",Income Statement::Personnel costs; Income Statement::Other costs,10
ALFA.ST,XSGA_COMPONENTS,"Candidate list clearly misses 'Sales costs', which is a core SG&A component shown above operating income. Because the explicit 'Selling, General & Admin' row exists but appears empty in the preview, I included it plus the closest overhead components to ensure coverage across years. Excluded STAFF EXPENSES to avoid likely overlap/double counting with sales/admin/R&D buckets. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Sales costs","Income Statement::Selling, General & Admin; Income Statement::Sales costs; Income Statement::Administration costs; Income Statement::Research and development costs",10
ALIFb.ST,REVT,"Used candidate-list label 'Total Revenue' and additionally included 'Net sales' from the full preview as a fallb